# ML_G0_P00040005_answer

## 0. 정답본 범위
- Gate: G0
- Block: B
- Phase: P0004 + P0005
- Topic: split/scaling + output/loss/metric 설계
- Problem notebook: ML_G0_P00040005.ipynb
- Correct answers included: Yes
- Output language: Korean
- Data directory: data/ML_G0_P00040005

## 1. 핵심 기준표

### 1-1. split/scaling 기준표

| 기준 | 정답 |
|---|---|
| split 순서 | 전처리 fit보다 split 먼저 |
| scaler.fit | train에만 적용 |
| validation/test | train에서 fit된 scaler로 transform만 적용 |
| encoder/imputer fit | train에만 적용 |
| stratify | 분류 문제에서 class 비율 유지를 위해 `stratify=y` 고려 |
| test set | 마지막 최종 평가에만 사용 |

### 1-2. leakage 기준표

| 항목 | 의미 | 예 |
|---|---|---|
| target leakage | 정답 또는 정답 이후 생성된 정보가 X에 들어감 | `final_score`, `final_grade_after_exam` |
| irrelevant ID feature | 식별 번호라 예측 원인으로 보기 어려움 | `student_id`, `house_id`, `customer_id`, `road_id` |
| preprocessing leakage | 전체 데이터로 scaler/encoder/imputer를 fit함 | split 전 `fit_transform` |
| test leakage | test를 반복 확인하며 모델을 고침 | test를 validation처럼 사용 |

### 1-3. 문제 유형별 output/loss/metric 기준표

| 문제 유형 | y 의미 | output | activation | loss | metric |
|---|---|---|---|---|---|
| 회귀 | 연속 수치값 | Dense(1) | 없음 또는 linear | MSE | MAE/RMSE |
| 이진분류 | 0/1 class | Dense(1) | sigmoid | Binary Cross Entropy | accuracy, precision, recall, F1 |
| 다중분류 | K개 class 중 하나 | Dense(K) | softmax | Cross Entropy | accuracy, macro-F1, confusion matrix |
| 순서형 class | 순서 있는 class | 기본은 Dense(K)+softmax | softmax | CE 또는 ordinal/cost-sensitive | macro-F1, confusion matrix, 평균 비용 |

### 1-4. sigmoid vs softmax 비교표

| 항목 | sigmoid | softmax |
|---|---|---|
| 입력 | logit 하나 또는 독립 logit 여러 개 | K개 logit |
| 출력 | 각 logit을 0~1 값으로 변환 | 합이 1인 확률분포 |
| 기본 사용 | 이진분류 `Dense(1)+sigmoid` | single-label multi-class `Dense(K)+softmax` |
| 해석 | `P(y=1|X)` | 각 class 확률 |
| 여러 개 사용 | multi-label 문제에 적합 | 상호배타적 다중분류에 적합 |
| 경쟁 구조 | 없음. 각 label이 독립 가능 | 있음. class들이 서로 경쟁 |

### 1-5. ordinal feature/target 처리 기준표

| 대상 | 상황 | 처리 기준 |
|---|---|---|
| ordinal feature | 순서와 간격 모두 의미 있음 | ordinal integer 가능 |
| ordinal feature | 순서는 있지만 간격이 애매함 | one-hot 또는 cumulative/thermometer encoding 고려 |
| ordinal feature | 선형모델/DNN/KNN/SVM | raw 0/1/2가 거리나 선형 효과로 해석될 수 있어 조심 |
| ordinal feature | Tree/Random Forest | threshold split이라 integer encoding이 상대적으로 자연스러울 수 있음 |
| ordinal code | normalization 적용 | 값 범위만 바뀌며 거리 가정은 사라지지 않음 |
| ordinal target | Gate 0 기본 접근 | 다중분류로 시작하고 평가/해석에서 순서 고려 |
| ordinal target | 심화 접근 | ordinal regression 또는 cost-sensitive classification |

### 1-6. loss vs metric 기준표

| 항목 | 의미 | 예 |
|---|---|---|
| loss | 학습 중 gradient 계산에 직접 쓰이며 모델이 줄이려는 목적함수 | MSE, BCE, CE |
| metric | 사람이 성능을 해석하고 모델을 비교하기 위해 보는 지표 | MAE, RMSE, accuracy, F1 |
| 회귀 | loss=MSE, metric=MAE/RMSE | 큰 오차 최적화와 원단위 해석 분리 |
| 이진분류 | loss=BCE, metric=accuracy/precision/recall/F1 | 확률 학습과 성능 해석 분리 |
| 다중분류 | loss=CE, metric=accuracy/macro-F1/confusion matrix | class 확률 학습과 class별 평가 분리 |

## 2. 데이터 확인 코드


In [ ]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path("data/ML_G0_P00040005")
assert DATA_DIR.exists()

exams = pd.read_csv(DATA_DIR / "student_exam_leakage.csv")
houses = pd.read_csv(DATA_DIR / "house_price_regression.csv")
customers = pd.read_csv(DATA_DIR / "customer_purchase_binary.csv")
traffic = pd.read_csv(DATA_DIR / "traffic_multiclass.csv")

assert exams.shape == (12, 8)
assert houses.shape == (12, 7)
assert customers.shape == (12, 9)
assert traffic.shape == (12, 8)
print("G0-B data ok")


## 2. 문항별 정답


### Q1. Leakage와 split/scaling 순서 감사

#### 정답
- 목표 y = `final_score`
- 정상 feature 후보 = `study_hours`, `attendance_rate`, `assignment_score`, `midterm_score`
- feature에서 제외해야 할 column = `student_id`, `final_score`, `final_grade_after_exam`, `teacher_comment_after_exam`
- target leakage가 되는 column = `final_score`, `final_grade_after_exam`, `teacher_comment_after_exam`
- irrelevant ID column = `student_id`
- target leakage와 irrelevant ID feature의 차이 = leakage는 정답이나 예측 시점 이후 정보를 X에 넣는 문제이고, irrelevant ID는 예측 원인으로 보기 어려운 식별자를 넣는 문제다.
- `final_grade_after_exam`, `teacher_comment_after_exam`이 위험한 이유 = `final_score` 이후에 만들어진 사후 정보라 실제 예측 시점에는 알 수 없고, target 정보를 우회적으로 담고 있다.
- `fit_transform`을 split 전에 하면 새는 정보 = validation/test의 평균, 표준편차, min/max 같은 분포 정보가 train 전처리 기준에 섞인다.
- 올바른 split/scaling 순서 = X/y 분리와 제외 column 결정 -> train/validation/test split -> scaler.fit은 train에만 -> validation/test는 transform만
- 문제 유형 = `final_score`라는 실제 수치값을 예측하는 회귀

#### 왜 그런지
모델은 예측 시점에 사용할 수 있는 정보만 X로 받아야 한다. 정답 자체 또는 정답 이후 생성된 정보가 들어가면 모델은 일반화 능력을 배운 것이 아니라 답안지를 본다. 반면 ID column은 정답을 직접 담지는 않지만, 번호 크기를 의미 있는 수치처럼 오해하게 만들 수 있어 제외한다.

#### 수식
StandardScaler의 평균과 표준편차는 다음 기준으로 계산된다.

```text
z = (x - mean) / std
```

전체 데이터로 mean/std를 계산하면 test 분포가 train 전처리에 반영된다. 따라서 `mean_train`, `std_train`만 사용해야 한다.

#### 자주 하는 오답
- `final_score`를 y로 쓰면서 X에도 넣는다.
- ID column을 leakage라고 부른다.
- split 전에 전체 데이터에 `fit_transform`한다.

#### 오답튜터 기준
먼저 “예측 시점에 이 정보를 알 수 있는가?”를 묻는다. 알 수 없다면 leakage 후보이고, 단순 식별자라면 irrelevant feature다.

#### 최종 답안형
`final_score` 예측은 회귀 문제다. X는 시험 전 알 수 있는 학습 관련 feature만 사용하고, `final_score` 및 시험 후 생성된 column은 leakage로 제외한다. `student_id`는 irrelevant ID로 제외한다. scaling은 split 후 train에만 fit하고 validation/test에는 transform만 적용한다.


### Q2. 회귀 pipeline과 output/loss/metric 설계

#### 정답
- X column = `area_m2`, `rooms`, `distance_station_km`, `building_age`, `tax_rate`
- y column = `price_million`
- feature에서 제외할 column = `house_id`
- 문제 유형 = 회귀
- scaling이 필요한 이유 = feature마다 단위와 범위가 달라 gradient 기반 학습에서 큰 스케일 feature가 과도하게 영향을 줄 수 있다.
- split 후 scaling 순서 = split 먼저 -> `scaler.fit(X_train)` -> train/validation/test에는 `transform` 적용
- 왜 Dense(1)인가 = 각 집마다 예측할 값이 집값 숫자 하나이기 때문이다.
- 왜 sigmoid/softmax를 쓰지 않는가 = 집값은 class 확률이 아니며 0~1 또는 합이 1인 확률분포로 제한하면 안 된다.
- 왜 loss로 MSE를 쓸 수 있는가 = 연속값 오차를 제곱해 큰 오차를 강하게 줄이도록 gradient를 만들 수 있기 때문이다.
- 왜 metric으로 MAE/RMSE를 볼 수 있는가 = 원래 y 단위로 평균 오차를 해석하기 쉽기 때문이다.
- target scaling을 했다면 최종 해석 방법 = scaled 예측값과 y를 `inverse_transform`으로 원래 단위로 되돌린 뒤 MAE/RMSE를 다시 계산한다.
- DNN output layer = `Dense(1)`
- activation = 없음 또는 linear
- loss = MSE
- metric = MAE 또는 RMSE

#### 왜 그런지
회귀는 실제 수치량을 예측한다. 출력값은 class 확률이 아니라 실수값이어야 하므로 마지막 activation은 보통 두지 않는다. MSE는 학습 중 큰 오차를 강하게 줄이기 좋고, MAE/RMSE는 사람이 원래 단위로 성능을 해석하기 좋다.

#### 수식
```text
MSE = (1/n) Σ (y - ŷ)^2
MAE = (1/n) Σ |y - ŷ|
RMSE = sqrt(MSE)
```

MSE가 큰 오차를 강하게 벌주는 이유:

```text
오차 2 -> 4
오차 10 -> 100
```

Gradient 관점:

```text
dMSE/dŷ = 2(ŷ - y) / n
```

즉 오차가 클수록 gradient도 커져 update에 더 강하게 반영된다.

단위 해석:
- MSE는 y 단위의 제곱 단위라 해석이 어렵다.
- RMSE는 `sqrt(MSE)`이므로 원래 y 단위로 돌아와 해석하기 쉽다.
- MAE는 처음부터 y와 같은 단위라 직관적이다.

Target scaling 해석:
- `transform`: 원래 단위 -> scaled 단위
- `inverse_transform`: scaled 단위 -> 원래 단위
- 학습 중 loss는 scaled 단위에서 계산될 수 있다.
- 최종 보고는 원래 단위로 되돌린 뒤 MAE/RMSE를 계산한다.

#### 자주 하는 오답
- 회귀인데 `sigmoid`를 붙여 출력 범위를 0~1로 제한한다.
- `house_id`를 feature로 사용한다.
- scaled target의 MAE를 원래 단위 오차처럼 해석한다.

#### 오답튜터 기준
“y가 실제 수치량인가, class label인가?”를 먼저 묻는다. 실제 수치량이면 확률 activation보다 linear output이 기본이다.

#### 최종 답안형
집값 예측은 회귀 문제다. 집값 숫자 하나를 예측하므로 `Dense(1)`을 쓰고, class 확률이 아니므로 activation은 없음 또는 linear로 둔다. loss는 큰 오차를 강하게 줄이는 MSE를 쓰고, metric은 원래 단위 해석이 쉬운 MAE/RMSE를 본다. target scaling을 했다면 최종 평가는 `inverse_transform` 후 원래 단위에서 계산한다.


### Q3. 이진분류와 class imbalance metric 설계

#### 정답
- X column = `age`, `income`, `city`, `device_type`, `visit_count`, `cart_amount`, `days_since_last_visit`
- y column = `purchased`
- feature에서 제외할 column = `customer_id`
- categorical column = `city`, `device_type`
- 필요한 전처리 = categorical column은 OneHotEncoding 등으로 숫자화하고, 수치 feature는 split 후 train 기준으로 scaling한다.
- 문제 유형 = 이진분류
- split에서 stratify가 필요한 이유 = train/validation/test에서 구매/미구매 class 비율이 크게 깨지는 것을 막기 위해서다.
- 왜 Dense(1)+sigmoid인가 = 구매 class `y=1`일 확률 하나를 출력하면 되기 때문이다.
- Binary Cross Entropy는 어떤 확률을 벌주는가 = 정답 class에 낮은 확률을 주는 예측을 크게 벌준다. y=1이면 p가 낮을 때, y=0이면 p가 높을 때 loss가 커진다.
- accuracy가 높은데도 모델이 나쁠 수 있는 상황 = 구매자가 매우 적을 때 전부 미구매로 예측해도 accuracy가 높게 보일 수 있다.
- precision, recall, F1의 차이 = precision은 구매라고 예측한 것의 정확도, recall은 실제 구매자를 얼마나 잡았는지, F1은 둘의 균형이다.
- F1이 산술평균이 아니라 조화평균인 이유 = precision 또는 recall 중 하나가 매우 낮으면 성능을 낮게 반영하기 위해서다.
- output/loss/metric = `Dense(1)`, `sigmoid`, Binary Cross Entropy, accuracy/precision/recall/F1

#### 왜 그런지
이진분류는 class 1의 확률 하나를 예측하면 충분하다. sigmoid는 logit 하나를 0~1 확률로 바꾸고, BCE는 정답 class 확률이 낮을수록 큰 벌점을 준다. 불균형 데이터에서는 accuracy만 보면 관심 class를 놓쳐도 좋은 모델처럼 보일 수 있다.

#### 수식
```text
Precision = TP / (TP + FP)
Recall = TP / (TP + FN)
F1 = 2PR / (P + R)
```

조화평균 예시:

```text
Precision = 1.0, Recall = 0.1
산술평균 = (1.0 + 0.1) / 2 = 0.55
F1 = 2 * 1.0 * 0.1 / (1.0 + 0.1) = 0.1818
```

F1은 한쪽만 높은 모델을 과대평가하지 않는다.

F1 평균 방식:
- macro-F1 = class별 F1을 단순 평균한다. 소수 class도 동일 비중이다.
- weighted-F1 = class별 F1을 sample 수로 가중평균한다.
- micro-F1 = 전체 TP/FP/FN을 합산해 계산한다. 단일 label 다중분류에서는 accuracy와 같아지는 경우가 많다.

#### 자주 하는 오답
- `city`, `device_type`을 문자열 그대로 모델에 넣는다.
- accuracy만 보고 구매 class recall을 놓친다.
- precision과 recall을 구분하지 못한다.

#### 오답튜터 기준
관심 class가 무엇인지 묻고, “모델이 전부 0으로 예측하면 어떤 metric이 높고 어떤 metric이 낮은가?”를 따져보게 한다.

#### 최종 답안형
구매 여부 예측은 0/1 이진분류다. `Dense(1)+sigmoid`로 구매 확률 하나를 출력하고 BCE로 정답 확률이 낮은 예측을 벌준다. class imbalance가 있으면 accuracy만으로 부족하므로 precision, recall, F1을 함께 본다.


### Q4. 다중분류/순서형 target과 softmax CE 설계

#### 정답
- X column = `hour`, `weekday`, `weather`, `vehicle_count`, `avg_speed`, `accident`
- y column = `congestion_level`
- feature에서 제외할 column = `road_id`
- categorical column = `weekday`, `weather`
- label mapping 보존이 필요한 이유 = ndarray/tensor로 바꾸면 0/1/2/3이 어떤 혼잡도 이름인지 자동으로 보존되지 않기 때문이다.
- 문제 유형 = 다중분류. 다만 원활 < 보통 < 혼잡 < 매우 혼잡 순서가 있으므로 순서형 target 성격도 있다.
- 왜 Dense(4)+softmax인가 = class가 4개이고 한 샘플은 그중 하나에만 속하므로 class별 logit 4개를 softmax 확률분포로 바꾼다.
- sigmoid 여러 개와 softmax의 차이 = sigmoid 여러 개는 각 label이 독립적으로 켜질 수 있는 multi-label에 적합하고, softmax는 상호배타적 single-label multi-class에 적합하다.
- sparse categorical cross entropy와 categorical cross entropy의 차이 = 모델 출력은 둘 다 `Dense(K)+softmax`지만, y가 정수 label이면 sparse, y가 one-hot이면 categorical을 쓴다.
- 순서형 target을 softmax 다중분류로 처리할 때의 장점 = 거리 가정 없이 class label로 안전하게 처리한다.
- 한계 = 0->1 오류와 0->3 오류의 심각도 차이를 CE loss가 직접 반영하지 않는다.
- cost matrix는 무엇을 보완하는가 = 실제 class와 예측 class가 멀수록 더 큰 비용을 부여해 순서형 오류 심각도를 반영한다.
- 순서형 feature 처리 = 간격이 의미 있으면 ordinal integer 가능, 간격이 애매하면 one-hot 또는 cumulative encoding을 고려한다.
- 정규화가 ordinal code의 거리 가정을 없애는가 = 아니다. 값 범위만 바꾸고 등간격 가정은 남는다.
- 기본 설계 = `Dense(4)`, `softmax`, sparse categorical cross entropy, accuracy/macro-F1/confusion matrix

#### 왜 그런지
`congestion_level`은 숫자지만 실제 양이 아니라 class code다. 4개 class 중 하나를 고르는 문제라 기본적으로 softmax 다중분류다. 그러나 class에 순서가 있으므로 평가와 해석에서는 멀리 틀린 오류를 더 심각하게 볼 수 있다.

#### 수식
Cross Entropy:

```text
CE = - Σ y_k log(p_k)
```

one-hot 정답이 `[0,0,1,0]`이면 정답 class만 남는다.

```text
CE = -log(p_true)
```

수치 예시:

```text
p_true = 0.90 -> CE = -log(0.90) ≈ 0.105
p_true = 0.10 -> CE = -log(0.10) ≈ 2.303
```

정답 class 확률이 높으면 loss가 낮고, 낮으면 loss가 크다.

Keras loss 구분:

```text
y가 one-hot이면 categorical_crossentropy
y가 정수 label이면 sparse_categorical_crossentropy
모델 출력은 둘 다 Dense(K)+softmax
차이는 y의 저장 형식과 loss가 y를 읽는 방식
```

Ordinal target 세 접근:

A. 기본 다중분류

```text
Dense(4) + softmax + sparse categorical cross entropy
장점: 거리 가정 없이 class label로 안전하게 처리
단점: 0->1 오류와 0->3 오류의 심각도 차이를 loss가 직접 반영하지 않음
```

B. ordinal regression / cumulative threshold

```text
P(y>0), P(y>1), P(y>2)처럼 threshold 문제로 바꿈
장점: 순서 정보 반영
단점: 구현이 일반 softmax보다 복잡
```

C. cost-sensitive classification

```text
softmax 다중분류는 유지하되, 예측 class가 실제 class에서 멀수록 더 큰 비용 부여
예: cost(i,j) = (i-j)^2
```

Cost matrix:

| 실제 \ 예측 | 원활 | 보통 | 혼잡 | 매우 혼잡 |
|---|---:|---:|---:|---:|
| 원활 | 0 | 1 | 4 | 9 |
| 보통 | 1 | 0 | 1 | 4 |
| 혼잡 | 4 | 1 | 0 | 1 |
| 매우 혼잡 | 9 | 4 | 1 | 0 |

#### 자주 하는 오답
- class가 4개인데 `Dense(1)`을 사용한다.
- 순서형 target이라는 이유만으로 바로 회귀로 처리한다.
- normalization으로 ordinal code의 거리 문제가 해결된다고 생각한다.

#### 오답튜터 기준
먼저 “숫자가 실제 수치량인가 class code인가?”를 묻고, 다음으로 “class 사이 순서가 있는가?”를 따로 묻는다.

#### 최종 답안형
혼잡도는 4개 class 중 하나를 예측하는 다중분류이며 순서형 성격도 있다. 기본 설계는 `Dense(4)+softmax+sparse categorical cross entropy`다. 이 방식은 거리 가정 없이 안전하지만 class 간 순서 오류 비용을 직접 반영하지 않으므로, 심화로 ordinal regression이나 cost-sensitive classification을 고려할 수 있다.


### Q5. validation/test와 compile/fit/evaluate 역할 구분

#### 정답
- train set의 역할 = 모델 파라미터를 실제로 학습하는 데이터
- validation set의 역할 = 학습 중 일반화 성능을 확인하고 모델/하이퍼파라미터 선택에 사용하는 데이터
- test set의 역할 = 모든 선택이 끝난 뒤 마지막 최종 평가에만 사용하는 데이터
- validation을 test처럼 반복 사용하면 생기는 문제 = test 기준으로 모델을 고치게 되어 test 성능이 더 이상 공정한 최종 평가가 아니다.
- Sequential/add의 역할 = layer를 쌓아 모델 구조를 정의한다.
- compile은 구조를 학습시키는 단계인가, 학습 기준을 붙이는 단계인가 = 구조를 학습시키는 단계가 아니라 optimizer/loss/metric이라는 학습 기준을 붙이는 단계다.
- model.compile()의 역할 = optimizer, loss, metric을 모델에 연결한다.
- fit 내부에서 반복되는 과정 = forward -> loss 계산 -> backward/gradient -> parameter update
- model.fit()의 역할 = train data로 위 과정을 반복해 학습한다.
- model.evaluate()의 역할 = 학습이 끝난 모델을 validation/test 데이터에서 평가한다.
- validation은 왜 test와 다른가 = validation은 실험 중 모델 선택에 쓰이고, test는 최종 보고 전에 숨겨두는 평가 데이터다.
- loss와 metric의 차이 = loss는 gradient 계산에 직접 쓰이며 모델이 줄이려는 목적함수이고, metric은 사람이 성능을 해석하고 비교하기 위해 보는 지표다.

#### 왜 그런지
모델 개발 과정에서 validation은 여러 번 볼 수 있지만 test는 마지막에 한 번만 봐야 한다. test를 반복해서 보면 모델 선택에 test 정보가 들어가 최종 성능이 낙관적으로 왜곡된다.

#### 수식과 예시
loss/metric 예:

```text
회귀: loss=MSE, metric=MAE/RMSE
이진분류: loss=Binary Cross Entropy, metric=accuracy, precision, recall, F1
다중분류: loss=Cross Entropy, metric=accuracy, macro-F1, confusion matrix
```

loss는 미분 가능한 목적함수로 gradient를 만든다. metric은 반드시 미분 가능할 필요가 없고, 사람이 해석하기 좋은 값이면 된다.

#### 자주 하는 오답
- `compile`이 학습을 수행한다고 생각한다.
- validation과 test를 같은 것으로 본다.
- metric을 loss와 같은 것으로 본다.

#### 오답튜터 기준
Keras 절차를 구조 설계, 학습 기준 연결, 학습 실행, 최종 평가로 나누어 말하게 한다.

#### 최종 답안형
`Sequential/add`는 구조 설계, `compile`은 optimizer/loss/metric 연결, `fit`은 forward/loss/backward/update 반복 학습, `evaluate`는 학습된 모델 평가다. validation은 실험 중 일반화 확인용이고 test는 마지막 최종 평가용이다. loss는 학습 목적함수이고 metric은 해석 지표다.


### Q6. 통합 설계 문제

#### 정답
세 프로젝트 중 하나를 선택하면 된다. 아래는 예시 답안이다.

##### 예시 B. customer_purchase_binary.csv로 purchased 예측

- 선택한 프로젝트 = B. 구매 여부 예측
- 문제 유형 = 이진분류
- X/y 분리 = X는 `age`, `income`, `city`, `device_type`, `visit_count`, `cart_amount`, `days_since_last_visit`, y는 `purchased`
- 제외할 column = `customer_id`
- encoding/scaling 계획 = `city`, `device_type`은 OneHotEncoding, 수치 feature는 split 후 train 기준으로 StandardScaler 또는 MinMaxScaler 적용
- split 계획 = train/validation/test로 나누고 class 비율 유지를 위해 `stratify=y`, 재현성을 위해 `random_state` 사용
- leakage 방지 원칙 = train에만 encoder/scaler를 fit하고 validation/test에는 transform만 적용한다. test는 마지막 평가에만 사용한다.
- DNN output/activation/loss/metric = `Dense(1)`, sigmoid, Binary Cross Entropy, accuracy/precision/recall/F1
- 왜 그 output/loss/metric을 고르는가 = y가 구매 여부 0/1이므로 class 1 확률 하나를 예측하면 되고, BCE는 정답 class 확률이 낮은 예측을 벌준다. 불균형 가능성이 있으므로 accuracy 외 precision/recall/F1을 본다.
- validation/test 사용 계획 = validation으로 모델 선택, test로 최종 성능 평가
- 최종 한 문장 요약 = DataFrame에서 X/y와 categorical 처리를 설계한 뒤 split을 먼저 하고, train 기준 전처리 fit을 적용해 `Dense(1)+sigmoid` 이진분류 모델을 학습·평가한다.

##### 회귀 프로젝트를 선택했다면 포함할 내용

`price_million`은 연속 수치값이므로 `Dense(1)+linear`, MSE, MAE/RMSE를 쓴다. target scaling을 했다면 예측값과 y를 `inverse_transform`으로 원래 단위로 되돌린 뒤 MAE/RMSE를 계산한다.

##### 순서형 target을 선택했다면 포함할 내용

`congestion_level`은 기본적으로 `Dense(4)+softmax+sparse CE` 다중분류로 시작한다. 다만 class 순서가 있으므로 confusion matrix나 평균 비용을 함께 보고, 심화로 ordinal regression 또는 cost-sensitive loss를 고려할 수 있다.

#### 왜 그런지
통합 설계에서는 X/y, 제외 column, encoding, scaling, split, leakage 방지, output/loss/metric이 모두 같은 문제 유형을 향해야 한다. 한 단계라도 문제 유형과 어긋나면 pipeline 전체가 흔들린다.

#### 자주 하는 오답
- 회귀/분류를 맞히고도 output/loss를 반대로 쓴다.
- split 전에 전체 데이터에 encoding/scaling fit을 한다.
- test를 여러 번 보며 모델을 고친다.

#### 오답튜터 기준
답안이 길어도 y의 의미, split/fit 순서, output/loss/metric의 일관성을 우선 채점한다.

#### 최종 답안형
선택한 프로젝트의 y 의미가 문제 유형을 결정한다. 그에 맞춰 X에서 ID/leakage column을 제외하고, categorical encoding과 scaling은 split 후 train에만 fit한다. output/activation/loss/metric은 회귀, 이진분류, 다중분류 중 해당 유형에 맞게 고르고, validation은 모델 선택, test는 최종 평가에만 사용한다.


## 3. 수식 부록

### 3-1. MSE / MAE / RMSE

```text
MSE = (1/n) Σ (y - ŷ)^2
MAE = (1/n) Σ |y - ŷ|
RMSE = sqrt(MSE)
```

MSE는 큰 오차를 강하게 벌준다.

```text
오차 2 -> 4
오차 10 -> 100
```

Gradient:

```text
dMSE/dŷ = 2(ŷ - y) / n
```

오차가 클수록 gradient가 커져 update에 더 강하게 반영된다.

### 3-2. Binary Cross Entropy

```text
BCE = - [ y log(p) + (1-y) log(1-p) ]
```

- y=1인데 p가 낮으면 loss가 커진다.
- y=0인데 p가 높으면 loss가 커진다.

### 3-3. Cross Entropy

```text
CE = - Σ y_k log(p_k)
```

one-hot 정답이 `[0,0,1,0]`이면:

```text
CE = -log(p_true)
```

```text
p_true = 0.90 -> CE ≈ 0.105
p_true = 0.10 -> CE ≈ 2.303
```

### 3-4. sigmoid / softmax

```text
sigmoid(z) = 1 / (1 + exp(-z))
softmax(z_k) = exp(z_k) / Σ exp(z_j)
```

sigmoid는 logit 하나를 0~1 확률 하나로 바꾸고, softmax는 K개 logit을 합이 1인 확률분포로 바꾼다.

### 3-5. precision / recall / F1

```text
Precision = TP / (TP + FP)
Recall = TP / (TP + FN)
F1 = 2PR / (P + R)
```

예:

```text
Precision=1.0, Recall=0.1
산술평균=0.55
F1=0.1818
```

F1은 한쪽만 높은 모델을 과대평가하지 않는다.

### 3-6. macro / weighted / micro F1

```text
macro-F1 = class별 F1을 단순 평균, 소수 class도 동일 비중
weighted-F1 = class별 F1을 sample 수로 가중평균
micro-F1 = 전체 TP/FP/FN을 합산해 계산
```

단일 label 다중분류에서는 micro-F1이 accuracy와 같아지는 경우가 많다.

### 3-7. cost matrix 평균 비용

순서형 class에서 예측 오류의 평균 비용을 볼 수 있다.

```text
cost(i,j) = (i-j)^2
평균 비용 = 전체 샘플의 cost(y_true, y_pred) 평균
```

| 실제 \ 예측 | 원활 | 보통 | 혼잡 | 매우 혼잡 |
|---|---:|---:|---:|---:|
| 원활 | 0 | 1 | 4 | 9 |
| 보통 | 1 | 0 | 1 | 4 |
| 혼잡 | 4 | 1 | 0 | 1 |
| 매우 혼잡 | 9 | 4 | 1 | 0 |


## 4. 다음 Block 연결

### G0-C 전통 ML 압축 + 종합 설계

G0-C에서는 지금까지의 기준을 더 짧은 전통 ML pipeline으로 압축한다.

```text
DataFrame -> X/y -> split -> encoding/scaling -> model -> metric -> leakage audit
```

### Gate 1 Perceptron/MLP/Gradient 연결

Gate 1에서는 왜 loss가 gradient를 만들고, activation과 output layer가 학습 가능한 함수 구조를 만드는지 더 직접적으로 다룬다.

연결 문장:

```text
G0-B에서 고른 loss는 Gate 1에서 gradient의 출발점이 된다.
G0-B에서 고른 output/activation은 Gate 1에서 모델 함수의 마지막 형태가 된다.
G0-B에서 분리한 train/validation/test는 Gate 1 이후 모든 실험의 기본 안전장치가 된다.
```
